# Train Markov bac 1 + GRU4Rec dung dac ta de cuong (RetailRocket)

Notebook nay tai hien dung giao thuc trong file *De cuong do an chuyen nganh*: dung chuoi `view` lam don vi nhan chinh, Markov bac 1 khong trong so, GRU4Rec huan luyen theo **session-parallel mini-batch** voi **loss TOP1** (dung ky thuat goc cua Hidasi et al., 2016), co tap validation rieng va early stopping theo MRR@20.

`category`/`event_type` (addtocart, transaction) **khong nam trong bai toan chinh** - chung chi duoc dung o **Phan mo rong** o cuoi notebook, nhu mot thi nghiem ablation rieng biet, khong thay doi ket qua cua mo hinh bat buoc (Most Popular, Markov, GRU4Rec).

## Chuan bi tren Kaggle
1. Tao notebook moi, upload file nay.
2. **Add Input** -> dataset `retailrocket/ecommerce-dataset` (khong can bat Internet).
3. Bat GPU (Settings > Accelerator) - khuyen nghi vi huan luyen co nhieu buoc.
4. Run All.

## Noi dung
1. Doc du lieu goc
2. Dung phien theo nguong khong hoat dong 30 phut (tren toan bo loai event, dung voi muc 3.5.2 de cuong)
3. Trich chuoi VIEW-only, chia Train/Validation/Test theo thoi gian (70/15/15), loai phien cat ngang ranh gioi
4. Xay tu dien tu TRAIN (co PAD, UNK)
5. Most Popular + Markov bac 1 (chi view, khong trong so, co kiem tra tong xac suat hang = 1)
6. GRU4Rec: session-parallel mini-batch, loss TOP1, Adagrad, UNK co dinh = 0, hidden_size = 100
7. Huan luyen co validation + early stopping theo MRR@20
8. Danh gia Hit Rate@K/MRR@K tren test - 2 lop: *candidate-only* (nhan trong tu dien train) va *toan bo query* (nhan ngoai tu dien tinh la truot)
9. Luu artifact CORE
10. **Phan mo rong**: GRU4Rec + category (tu `item_properties`) nhu mot ablation rieng

In [ ]:
import glob
import os
import pickle
import random
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dang chay tren:", DEVICE)

PAD_IDX = 0
UNK_IDX = 1
SESSION_GAP_MS = 30 * 60 * 1000  # nguong khong hoat dong 30 phut
HIDDEN_DIM = 100                 # dung hidden_size=100 nhu de cuong muc 3.6.3
BATCH_SIZE = 50                  # 50 phien hoat dong song song (session-parallel mini-batch)
LR = 0.01
MAX_EPOCHS = 30
EARLY_STOP_PATIENCE = 3
EARLY_STOP_EPS = 1e-4
K_VALUES = (5, 10, 20)

## Buoc 1 - Doc du lieu goc tu Kaggle Input

In [ ]:
def find_file(filename):
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if not matches:
        raise FileNotFoundError(
            f"Khong tim thay {filename} trong /kaggle/input - "
            "hay chac chan ban da Add Input dataset 'retailrocket/ecommerce-dataset'."
        )
    return matches[0]


EVENTS_PATH = find_file("events.csv")
print("events.csv:", EVENTS_PATH)

events_df = pd.read_csv(EVENTS_PATH)
events_df["itemid"] = events_df["itemid"].astype(str)
events_df["visitorid"] = events_df["visitorid"].astype(str)

# phan hoa thu tu khi trung timestamp bang so dong goc (dung muc 3.3.1 de cuong)
events_df = events_df.reset_index().rename(columns={"index": "orig_row"})

print("So luong event:", len(events_df))
print(events_df["event"].value_counts())

## Buoc 2 - Dung phien theo nguong khong hoat dong

Ranh gioi phien duoc xac dinh tren **toan bo loai event** (view/addtocart/transaction deu tinh vao thoi gian hoat dong), dung muc 3.5.2 cua de cuong: *"Addtocart va transaction duoc luu va cap nhat hoat dong phien nhung khong tang context_seq cua chuoi view"*. Chi khi trich chuoi du doan (Buoc 3) moi loc rieng lai cac lot `view`.

In [ ]:
events_df = events_df.sort_values(["visitorid", "timestamp", "orig_row"]).reset_index(drop=True)

prev_ts = events_df.groupby("visitorid")["timestamp"].shift(1)
gap = events_df["timestamp"] - prev_ts
is_new_session = gap.isna() | (gap > SESSION_GAP_MS)
session_seq = is_new_session.groupby(events_df["visitorid"]).cumsum()
events_df["session_id"] = events_df["visitorid"] + "_" + session_seq.astype(str)

print("So luong session:", events_df["session_id"].nunique())

## Buoc 3 - Trich chuoi VIEW-only va chia Train/Validation/Test theo thoi gian

Chia theo **do dai khoang thoi gian** (khong phai theo so dong), ty le 70/15/15, dung muc 3.2.1. Mot phien duoc gan vao mot tap chi khi TOAN BO su kien cua no nam gon trong khoang thoi gian tuong ung; phien co su kien o ca hai phia ranh gioi bi **loai rieng** va bao cao so luong.

In [ ]:
t_min = events_df["timestamp"].min()
t_max = events_df["timestamp"].max()
span = t_max - t_min
t1 = t_min + span * 0.70
t2 = t_min + span * 0.85

session_min_ts = events_df.groupby("session_id")["timestamp"].min()
session_max_ts = events_df.groupby("session_id")["timestamp"].max()


def assign_split(session_id):
    lo, hi = session_min_ts[session_id], session_max_ts[session_id]
    if hi < t1:
        return "train"
    if lo >= t1 and hi < t2:
        return "validation"
    if lo >= t2:
        return "test"
    return "boundary"


split_of = {sid: assign_split(sid) for sid in events_df["session_id"].unique()}
print("Phan bo session theo tap:", Counter(split_of.values()))

In [ ]:
view_sessions = {"train": [], "validation": [], "test": []}

for session_id, group in events_df.groupby("session_id"):
    split = split_of[session_id]
    if split == "boundary":
        continue
    view_group = group[group["event"] == "view"].sort_values(["timestamp", "orig_row"])
    items = view_group["itemid"].tolist()
    if len(items) >= 2:
        view_sessions[split].append(items)

for split in ("train", "validation", "test"):
    n_queries = sum(len(s) - 1 for s in view_sessions[split])
    print(f"{split}: {len(view_sessions[split])} session (>=2 view) | {n_queries} tien to-nhan")

## Buoc 4 - Xay tu dien san pham (chi tu TRAIN)

`item2idx["<pad>"] = 0`, `item2idx["<unk>"] = 1`. San pham xuat hien o validation/test nhung khong co trong train se duoc anh xa ve `<unk>` khi encode - khong bi bo qua.

In [ ]:
item2idx = {"<pad>": 0, "<unk>": 1}
for session in view_sessions["train"]:
    for item_id in session:
        if item_id not in item2idx:
            item2idx[item_id] = len(item2idx)

idx2item = {idx: item for item, idx in item2idx.items() if item not in ("<pad>", "<unk>")}
candidate_items = set(idx2item.values())  # tap ung vien C - dung chung cho moi phuong phap
VOCAB_SIZE = len(item2idx)

print("Kich thuoc tu dien (vocab):", VOCAB_SIZE)


def encode_item(item_id):
    return item2idx.get(item_id, UNK_IDX)


train_sessions_idx = [[encode_item(i) for i in s] for s in view_sessions["train"]]

## Buoc 5 - Most Popular va Markov bac 1

Ca hai deu la thong ke tren TRAIN, khong dung optimizer/epoch, khong lam tron (dung muc 2.3, 3.4.1-3.4.2). Markov chi dem chuyen tiep giua **hai lot view lien tiep trong cung phien**.

In [ ]:
view_counter = Counter()
transitions = defaultdict(lambda: defaultdict(int))

for session in view_sessions["train"]:
    for item in session:
        view_counter[item] += 1
    for a, b in zip(session[:-1], session[1:]):
        transitions[a][b] += 1

global_popularity = [item for item, _ in view_counter.most_common()]

markov_matrix = {}
for a, counts in transitions.items():
    total = sum(counts.values())
    markov_matrix[a] = {b: c / total for b, c in counts.items()}

# kiem tra tong xac suat moi hang ~ 1 (phat hien loi dung phien/dem chuyen tiep)
max_row_error = max(abs(sum(row.values()) - 1.0) for row in markov_matrix.values())
print(f"Markov matrix: {len(markov_matrix)} san pham nguon | sai so tong hang toi da: {max_row_error:.2e}")
print(f"Most Popular: {len(global_popularity)} san pham")


def predict_markov(history_items, k):
    last_item = history_items[-1]
    preds = []
    if last_item in markov_matrix:
        ranked = sorted(markov_matrix[last_item].items(), key=lambda kv: kv[1], reverse=True)
        preds = [item for item, _ in ranked]
    if len(preds) < k:
        seen = set(preds)
        for item in global_popularity:
            if item not in seen:
                preds.append(item)
                seen.add(item)
            if len(preds) >= k:
                break
    return preds[:k]


def predict_most_popular(history_items, k):
    return global_popularity[:k]

## Buoc 6 - GRU4Rec dung dac ta: session-parallel mini-batch + loss TOP1

Kien truc: 1 tang GRU mot chieu, hidden_size = 100, dau vao la embedding (tuong duong phep chieu one-hot qua ma tran trong so), PAD duoc che khoi hoc (padding_idx=0), UNK co vector co dinh bang 0 (khong hoc) - dung muc 3.4.3.

Huan luyen theo **session-parallel mini-batch**: moi buoc xu ly dong thoi 50 "cho" (slot), moi slot giu mot phien dang hoat dong; khi mot phien ket thuc, slot do duoc nap phien moi va reset trang thai an ve 0 - dung ky thuat goc cua GRU4Rec (Hidasi et al., 2016), khong phai mau (lich su, nhan) xao tron doc lap.

In [ ]:
class GRU4Rec(nn.Module):
    def __init__(self, vocab_size, hidden_dim=HIDDEN_DIM):
        super().__init__()
        self.item_emb = nn.Embedding(vocab_size, hidden_dim, padding_idx=PAD_IDX)
        self.cell = nn.GRUCell(hidden_dim, hidden_dim)
        self.output = nn.Linear(hidden_dim, vocab_size)
        with torch.no_grad():
            self.item_emb.weight[UNK_IDX].zero_()

    def forward_step(self, item_idx, hidden):
        x = self.item_emb(item_idx)
        h_new = self.cell(x, hidden)
        scores = self.output(h_new)
        return scores, h_new

In [ ]:
_neg_rng = random.Random(SEED)


def top1_loss(scores, targets, active_mask, vocab_size):
    """Loss TOP1 (cong thuc 2.18): mau am la nhan cua cac phien khac dang hoat dong
    trong CUNG buoc huan luyen; neu khong co mau am hop le, du phong lay toi da 20
    san pham ngau nhien tu tu dien train (co seed)."""
    active_idx = active_mask.nonzero(as_tuple=True)[0]
    A = active_idx.numel()
    if A == 0:
        return torch.zeros((), device=scores.device, requires_grad=True)

    active_scores = scores[active_idx]
    active_targets = targets[active_idx]

    pos_scores = active_scores.gather(1, active_targets.view(-1, 1))
    neg_scores = active_scores[:, active_targets]

    same_label = active_targets.unsqueeze(0) == active_targets.unsqueeze(1)
    self_mask = torch.eye(A, dtype=torch.bool, device=scores.device)
    exclude = same_label | self_mask
    valid_counts = (~exclude).sum(dim=1)

    terms = torch.sigmoid(neg_scores - pos_scores) + torch.sigmoid(neg_scores ** 2)
    terms = terms.masked_fill(exclude, 0.0)

    per_example = torch.zeros(A, device=scores.device)
    has_neg = valid_counts > 0
    if has_neg.any():
        per_example[has_neg] = terms[has_neg].sum(dim=1) / valid_counts[has_neg].float()

    no_neg_idx = (~has_neg).nonzero(as_tuple=True)[0]
    for local_i in no_neg_idx.tolist():
        global_i = active_idx[local_i].item()
        pos_item = active_targets[local_i].item()
        neg_items = []
        while len(neg_items) < 20:
            cand = _neg_rng.randint(2, vocab_size - 1)  # bo qua PAD=0, UNK=1
            if cand != pos_item:
                neg_items.append(cand)
        neg_t = torch.tensor(neg_items, device=scores.device)
        neg_s = scores[global_i, neg_t]
        pos_s = scores[global_i, pos_item]
        per_example[local_i] = (torch.sigmoid(neg_s - pos_s) + torch.sigmoid(neg_s ** 2)).mean()

    return per_example.mean()

In [ ]:
def train_epoch(model, optimizer, sessions_idx, hidden_dim=HIDDEN_DIM, batch_size=BATCH_SIZE):
    order = list(range(len(sessions_idx)))
    random.shuffle(order)
    n_sessions = len(order)

    slot_session = [-1] * batch_size
    slot_pos = [0] * batch_size
    hidden = torch.zeros(batch_size, hidden_dim, device=DEVICE)
    next_ptr = 0

    def fill_slot(slot):
        nonlocal next_ptr
        if next_ptr < n_sessions:
            slot_session[slot] = order[next_ptr]
            slot_pos[slot] = 0
            hidden[slot].zero_()
            next_ptr += 1
            return True
        slot_session[slot] = -1
        return False

    for slot in range(batch_size):
        fill_slot(slot)

    model.train()
    total_loss, n_steps = 0.0, 0

    while any(s != -1 for s in slot_session):
        input_items, target_items, active_mask = [], [], []
        for slot in range(batch_size):
            s_idx = slot_session[slot]
            if s_idx == -1:
                input_items.append(PAD_IDX)
                target_items.append(PAD_IDX)
                active_mask.append(False)
                continue
            session = sessions_idx[s_idx]
            pos = slot_pos[slot]
            input_items.append(session[pos])
            target_items.append(session[pos + 1])
            active_mask.append(True)

        active_t = torch.tensor(active_mask, device=DEVICE)
        input_t = torch.tensor(input_items, dtype=torch.long, device=DEVICE)
        target_t = torch.tensor(target_items, dtype=torch.long, device=DEVICE)

        scores, new_hidden = model.forward_step(input_t, hidden)
        loss = top1_loss(scores, target_t, active_t, VOCAB_SIZE)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            model.item_emb.weight[UNK_IDX].zero_()  # UNK giu co dinh = 0

        total_loss += loss.item()
        n_steps += 1
        hidden = new_hidden.detach()  # cat gradient qua buoc (truncated BPTT tung buoc)

        for slot in range(batch_size):
            if not active_mask[slot]:
                continue
            slot_pos[slot] += 1
            session = sessions_idx[slot_session[slot]]
            if slot_pos[slot] >= len(session) - 1:
                fill_slot(slot)

    return total_loss / max(n_steps, 1)

## Buoc 7 - Danh gia (dung cho ca validation va test) + vong lap huan luyen voi early stopping

In [ ]:
def gru_predict_all_prefixes(model, session_original_ids, hidden_dim=HIDDEN_DIM, k=max(K_VALUES)):
    """Chay 1 luot tuan tu qua ca phien (trang thai an bat dau = 0), tra ve
    danh sach (nhan_that, danh_sach_du_doan) cho MOI tien to trong phien -
    tuong duong voi viec chay lai tu dau cho tung tien to (dung muc 3.4.3)."""
    model.eval()
    hidden = torch.zeros(1, hidden_dim, device=DEVICE)
    results = []
    with torch.no_grad():
        for t in range(len(session_original_ids) - 1):
            cur_idx = torch.tensor([encode_item(session_original_ids[t])], device=DEVICE)
            scores, hidden = model.forward_step(cur_idx, hidden)
            scores = scores.clone()
            scores[0, PAD_IDX] = float("-inf")
            scores[0, UNK_IDX] = float("-inf")
            topk_idx = torch.topk(scores, k=min(k, scores.shape[-1]), dim=-1).indices[0].tolist()
            preds = [idx2item[i] for i in topk_idx if i in idx2item]
            results.append((session_original_ids[t + 1], preds))
    return results


def evaluate_tabular(predict_fn, sessions, k_values=K_VALUES):
    """Dung cho Markov/Most Popular: predict_fn(history_items, k) -> danh sach item goc."""
    queries = []
    for session in sessions:
        for t in range(1, len(session)):
            queries.append((session[:t], session[t]))
    return _aggregate(
        [(label, predict_fn(history, max(k_values))) for history, label in queries], k_values,
    )


def evaluate_gru(model, sessions, k_values=K_VALUES):
    all_results = []
    for session in sessions:
        all_results.extend(gru_predict_all_prefixes(model, session, k=max(k_values)))
    return _aggregate(all_results, k_values)


def _aggregate(label_pred_pairs, k_values):
    n_total = len(label_pred_pairs)
    n_in_candidate = sum(1 for label, _ in label_pred_pairs if label in candidate_items)
    oov_rate = 1 - (n_in_candidate / n_total) if n_total else 0.0

    report = {"n_queries": n_total, "oov_rate": oov_rate}
    for scope, filter_fn in (("candidate_only", lambda lb: lb in candidate_items), ("all_queries", lambda lb: True)):
        subset = [(label, preds) for label, preds in label_pred_pairs if filter_fn(label)]
        n = len(subset)
        for k in k_values:
            hits = sum(1 for label, preds in subset if label in preds[:k])
            report[f"{scope}_HitRate@{k}"] = round(hits / n, 4) if n else 0.0
        rr_sum = 0.0
        max_k = max(k_values)
        for label, preds in subset:
            preds_k = preds[:max_k]
            if label in preds_k:
                rr_sum += 1 / (preds_k.index(label) + 1)
        report[f"{scope}_MRR@{max_k}"] = round(rr_sum / n, 4) if n else 0.0
    return report

In [ ]:
model = GRU4Rec(VOCAB_SIZE).to(DEVICE)
optimizer = torch.optim.Adagrad(model.parameters(), lr=LR)

best_val_mrr = -1.0
best_state = None
epochs_no_improve = 0

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss = train_epoch(model, optimizer, train_sessions_idx)
    val_report = evaluate_gru(model, view_sessions["validation"])
    val_mrr20 = val_report["candidate_only_MRR@20"]

    print(f"epoch {epoch}/{MAX_EPOCHS} - loss {train_loss:.4f} - val MRR@20 (candidate) {val_mrr20:.4f}")

    if val_mrr20 > best_val_mrr + EARLY_STOP_EPS:
        best_val_mrr = val_mrr20
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"Dung som: {EARLY_STOP_PATIENCE} epoch lien tiep khong cai thien MRR@20 > {EARLY_STOP_EPS}")
            break

if best_state is not None:
    model.load_state_dict(best_state)
model.eval()
print("Da khoi phuc checkpoint tot nhat (val MRR@20 =", round(best_val_mrr, 4), ")")

## Buoc 8 - Danh gia tren TEST (2 lop: candidate-only va toan bo query)

In [ ]:
test_sessions = view_sessions["test"]

reports = {
    "Most Popular": evaluate_tabular(predict_most_popular, test_sessions),
    "Markov bac 1 (+ fallback Most Popular)": evaluate_tabular(predict_markov, test_sessions),
    "GRU4Rec (dac ta goc)": evaluate_gru(model, test_sessions),
}

rows = []
for name, report in reports.items():
    row = {"method": name, "n_queries": report["n_queries"], "oov_rate": report["oov_rate"]}
    for k in K_VALUES:
        row[f"HR@{k} (candidate)"] = report[f"candidate_only_HitRate@{k}"]
        row[f"HR@{k} (all)"] = report[f"all_queries_HitRate@{k}"]
    row["MRR@20 (candidate)"] = report["candidate_only_MRR@20"]
    row["MRR@20 (all)"] = report["all_queries_MRR@20"]
    rows.append(row)

comparison_df = pd.DataFrame(rows)
comparison_df

## Buoc 9 - Luu artifact CORE

In [ ]:
OUT_DIR = "/kaggle/working"

with open(f"{OUT_DIR}/markov.pkl", "wb") as f:
    pickle.dump(markov_matrix, f)
with open(f"{OUT_DIR}/global_popularity.pkl", "wb") as f:
    pickle.dump(global_popularity, f)
with open(f"{OUT_DIR}/item2idx.pkl", "wb") as f:
    pickle.dump(item2idx, f)
torch.save(model.state_dict(), f"{OUT_DIR}/gru4rec_core.pt")
comparison_df.to_csv(f"{OUT_DIR}/eval_core.csv", index=False)

print("Da luu artifact CORE vao", OUT_DIR)
print(os.listdir(OUT_DIR))

---
# PHAN MO RONG (khong bat buoc): GRU4Rec + category

Phan nay dung muc 1.2.2 va 5.2 cua de cuong: *"Trong pham vi mo rong cua do an, danh muc hoac loai su kien co the duoc xem xet lam du lieu ngu canh... Day la dinh huong can thuc nghiem, chua phai ket luan ve hieu qua cua thong tin bo tro."*

Day la **mot ablation rieng**, so sanh THEM voi ket qua CORE o tren - khong thay the hay lam thay doi Markov/GRU4Rec bat buoc. Chuoi su kien van CHI dung `view`; category chi duoc ghep them lam dac trung phu tro.

In [ ]:
CATEGORY_TREE_PATH = find_file("category_tree.csv")
ITEM_PROPS_PATHS = [find_file("item_properties_part1.csv"), find_file("item_properties_part2.csv")]

category_frames = []
for path in ITEM_PROPS_PATHS:
    chunk = pd.read_csv(path)
    chunk = chunk[chunk["property"] == "categoryid"]
    category_frames.append(chunk[["timestamp", "itemid", "value"]])

category_raw = pd.concat(category_frames, ignore_index=True)
category_raw["itemid"] = category_raw["itemid"].astype(str)
category_raw = category_raw.sort_values("timestamp")
item_category = category_raw.groupby("itemid")["value"].last().to_dict()
del category_frames, category_raw

cat2idx = {"<pad>": 0, "<unk>": 1}
for item_id in item2idx:
    if item_id in ("<pad>", "<unk>"):
        continue
    cat_id = item_category.get(item_id, "<unk>")
    if cat_id not in cat2idx:
        cat2idx[cat_id] = len(cat2idx)

CAT_VOCAB_SIZE = len(cat2idx)
print("So category (vocab):", CAT_VOCAB_SIZE)


def encode_cat(item_id):
    return cat2idx.get(item_category.get(item_id, "<unk>"), cat2idx["<unk>"])

In [ ]:
class GRU4RecCat(nn.Module):
    def __init__(self, vocab_size, cat_vocab_size, item_dim=84, cat_dim=16, hidden_dim=HIDDEN_DIM):
        super().__init__()
        self.item_emb = nn.Embedding(vocab_size, item_dim, padding_idx=PAD_IDX)
        self.cat_emb = nn.Embedding(cat_vocab_size, cat_dim, padding_idx=0)
        self.cell = nn.GRUCell(item_dim + cat_dim, hidden_dim)
        self.output = nn.Linear(hidden_dim, vocab_size)
        with torch.no_grad():
            self.item_emb.weight[UNK_IDX].zero_()

    def forward_step(self, item_idx, cat_idx, hidden):
        x = torch.cat([self.item_emb(item_idx), self.cat_emb(cat_idx)], dim=-1)
        h_new = self.cell(x, hidden)
        scores = self.output(h_new)
        return scores, h_new


def train_epoch_cat(model, optimizer, sessions_idx, sessions_cat_idx, hidden_dim=HIDDEN_DIM, batch_size=BATCH_SIZE):
    order = list(range(len(sessions_idx)))
    random.shuffle(order)
    n_sessions = len(order)

    slot_session = [-1] * batch_size
    slot_pos = [0] * batch_size
    hidden = torch.zeros(batch_size, hidden_dim, device=DEVICE)
    next_ptr = 0

    def fill_slot(slot):
        nonlocal next_ptr
        if next_ptr < n_sessions:
            slot_session[slot] = order[next_ptr]
            slot_pos[slot] = 0
            hidden[slot].zero_()
            next_ptr += 1
            return True
        slot_session[slot] = -1
        return False

    for slot in range(batch_size):
        fill_slot(slot)

    model.train()
    total_loss, n_steps = 0.0, 0

    while any(s != -1 for s in slot_session):
        input_items, input_cats, target_items, active_mask = [], [], [], []
        for slot in range(batch_size):
            s_idx = slot_session[slot]
            if s_idx == -1:
                input_items.append(PAD_IDX)
                input_cats.append(0)
                target_items.append(PAD_IDX)
                active_mask.append(False)
                continue
            session = sessions_idx[s_idx]
            cats = sessions_cat_idx[s_idx]
            pos = slot_pos[slot]
            input_items.append(session[pos])
            input_cats.append(cats[pos])
            target_items.append(session[pos + 1])
            active_mask.append(True)

        active_t = torch.tensor(active_mask, device=DEVICE)
        item_t = torch.tensor(input_items, dtype=torch.long, device=DEVICE)
        cat_t = torch.tensor(input_cats, dtype=torch.long, device=DEVICE)
        target_t = torch.tensor(target_items, dtype=torch.long, device=DEVICE)

        scores, new_hidden = model.forward_step(item_t, cat_t, hidden)
        loss = top1_loss(scores, target_t, active_t, VOCAB_SIZE)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            model.item_emb.weight[UNK_IDX].zero_()

        total_loss += loss.item()
        n_steps += 1
        hidden = new_hidden.detach()

        for slot in range(batch_size):
            if not active_mask[slot]:
                continue
            slot_pos[slot] += 1
            session = sessions_idx[slot_session[slot]]
            if slot_pos[slot] >= len(session) - 1:
                fill_slot(slot)

    return total_loss / max(n_steps, 1)


def gru_cat_predict_all_prefixes(model, session_original_ids, k=max(K_VALUES)):
    model.eval()
    hidden = torch.zeros(1, HIDDEN_DIM, device=DEVICE)
    results = []
    with torch.no_grad():
        for t in range(len(session_original_ids) - 1):
            item_id = session_original_ids[t]
            item_t = torch.tensor([encode_item(item_id)], device=DEVICE)
            cat_t = torch.tensor([encode_cat(item_id)], device=DEVICE)
            scores, hidden = model.forward_step(item_t, cat_t, hidden)
            scores = scores.clone()
            scores[0, PAD_IDX] = float("-inf")
            scores[0, UNK_IDX] = float("-inf")
            topk_idx = torch.topk(scores, k=min(k, scores.shape[-1]), dim=-1).indices[0].tolist()
            preds = [idx2item[i] for i in topk_idx if i in idx2item]
            results.append((session_original_ids[t + 1], preds))
    return results


def evaluate_gru_cat(model, sessions, k_values=K_VALUES):
    all_results = []
    for session in sessions:
        all_results.extend(gru_cat_predict_all_prefixes(model, session, k=max(k_values)))
    return _aggregate(all_results, k_values)

In [ ]:
train_sessions_cat_idx = [[encode_cat(i) for i in s] for s in view_sessions["train"]]

model_cat = GRU4RecCat(VOCAB_SIZE, CAT_VOCAB_SIZE).to(DEVICE)
optimizer_cat = torch.optim.Adagrad(model_cat.parameters(), lr=LR)

best_val_mrr_cat, best_state_cat, no_improve_cat = -1.0, None, 0

for epoch in range(1, MAX_EPOCHS + 1):
    loss = train_epoch_cat(model_cat, optimizer_cat, train_sessions_idx, train_sessions_cat_idx)
    val_report = evaluate_gru_cat(model_cat, view_sessions["validation"])
    val_mrr20 = val_report["candidate_only_MRR@20"]
    print(f"[mo rong] epoch {epoch}/{MAX_EPOCHS} - loss {loss:.4f} - val MRR@20 {val_mrr20:.4f}")

    if val_mrr20 > best_val_mrr_cat + EARLY_STOP_EPS:
        best_val_mrr_cat = val_mrr20
        best_state_cat = {k: v.clone() for k, v in model_cat.state_dict().items()}
        no_improve_cat = 0
    else:
        no_improve_cat += 1
        if no_improve_cat >= EARLY_STOP_PATIENCE:
            print("Dung som (mo rong)")
            break

if best_state_cat is not None:
    model_cat.load_state_dict(best_state_cat)
model_cat.eval()

In [ ]:
cat_report = evaluate_gru_cat(model_cat, test_sessions)
row = {"method": "[MO RONG] GRU4Rec + category", "n_queries": cat_report["n_queries"], "oov_rate": cat_report["oov_rate"]}
for k in K_VALUES:
    row[f"HR@{k} (candidate)"] = cat_report[f"candidate_only_HitRate@{k}"]
    row[f"HR@{k} (all)"] = cat_report[f"all_queries_HitRate@{k}"]
row["MRR@20 (candidate)"] = cat_report["candidate_only_MRR@20"]
row["MRR@20 (all)"] = cat_report["all_queries_MRR@20"]

extension_df = pd.concat([comparison_df, pd.DataFrame([row])], ignore_index=True)
extension_df

In [ ]:
torch.save(model_cat.state_dict(), f"{OUT_DIR}/gru4rec_ext_category.pt")
with open(f"{OUT_DIR}/cat2idx.pkl", "wb") as f:
    pickle.dump(cat2idx, f)
with open(f"{OUT_DIR}/item2category.pkl", "wb") as f:
    pickle.dump(item_category, f)
extension_df.to_csv(f"{OUT_DIR}/eval_with_extension.csv", index=False)

print("Da luu them artifact phan mo rong.")
print(os.listdir(OUT_DIR))

## Ket luan

- Phan CORE (Most Popular, Markov bac 1, GRU4Rec) tai hien dung don vi nhan (chi `view`), cach chia tap (70/15/15 theo thoi gian, loai phien cat ranh gioi), kien truc va qua trinh huan luyen (session-parallel mini-batch, loss TOP1, Adagrad, UNK co dinh, early stopping theo MRR@20 validation) nhu de cuong mo ta o Chuong 2-3.
- Phan MO RONG (category) la mot ablation rieng, khong thay doi ket qua CORE.
- **Don gian hoa da ghi nhan so voi dac ta day du** (can neu trong bao cao):
  - Chua trien khai day du chuoi su kien chuan hoa (schema_version, event_seq, run_id...) va tang Kafka/Spark/Redis/Cassandra that - day la notebook huan luyen ngoai tuyen, phan tich hop he thong (muc 3.5-3.7) thuoc ve ma nguon chinh cua du an, khong thuoc notebook nay.
  - Category duoc lay theo "gia tri quan sat moi nhat" tren toan bo log `item_properties`, chua dam bao dung thoi diem du doan (nhu luu y o muc 2.2.3) - chi anh huong Phan mo rong, khong anh huong CORE.
  - Markov chua tinh rieng truong `score_type` cho tung vi tri ket qua (phan biet diem Markov that voi diem du phong Most Popular) - hien tai chi tra ve danh sach da gop, can bo sung khi tich hop vao API phuc vu.

File can tai ve va dat vao `models/` cua project: `markov.pkl`, `global_popularity.pkl`, `item2idx.pkl`, `gru4rec_core.pt`, va (neu dung phan mo rong) `gru4rec_ext_category.pt`, `cat2idx.pkl`, `item2category.pkl`.

**Luu y tich hop:** kien truc GRU4Rec (dung GRUCell, chay tung buoc) khac voi `models/gru4rec.py` hien tai cua he thong (dung `nn.GRU` tren ca chuoi). File `streaming/inference.py` can duoc cap nhat de khop dung cach nap/goi model moi truoc khi dua vao phuc vu that.